# Global Poverty & Economic Inequality Dashboard Report

**Created by Hieu Nguyen**

This professional notebook report builds a clean dashboard from `global_poverty_economic_inequality.csv`, documents the full reproducible workflow, and turns the dataset into decision-ready insights.

The analysis focuses on poverty pressure, inequality concentration, human development, infrastructure access, and strategic country archetypes. Associations are interpreted carefully as analytical signals, not causal proof.


## 1. Environment Setup

The notebook starts by installing the required modules into the active kernel. In this project, the intended kernel is the local `venv` kernel named **Python (Global Poverty venv)**.


In [1]:
import os
from pathlib import Path

Path(".tmp").mkdir(exist_ok=True)
Path(".pip-cache").mkdir(exist_ok=True)
os.environ["TEMP"] = str(Path(".tmp").resolve())
os.environ["TMP"] = str(Path(".tmp").resolve())
os.environ["PIP_CACHE_DIR"] = str(Path(".pip-cache").resolve())

%pip install -q -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


## 2. Imports, Theme, and Helpers


In [2]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

COLORWAY = [
    "#0E7C86",  # teal
    "#D95F59",  # coral
    "#F2B84B",  # gold
    "#5B5F97",  # muted indigo
    "#2A9D8F",  # green teal
    "#8E6C88",  # mauve
    "#607D8B",  # blue gray
    "#E76F51",  # warm accent
]
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = COLORWAY


def fmt_pct(value: float, digits: int = 1) -> str:
    return "n/a" if pd.isna(value) else f"{value:.{digits}f}%"


def fmt_num(value: float, digits: int = 1) -> str:
    return "n/a" if pd.isna(value) else f"{value:,.{digits}f}"


def fmt_money(value: float) -> str:
    return "n/a" if pd.isna(value) else f"${value:,.0f}"


def minmax(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    series = pd.to_numeric(series, errors="coerce")
    span = series.max() - series.min()
    if pd.isna(span) or span == 0:
        scaled = pd.Series(0.5, index=series.index)
    else:
        scaled = (series - series.min()) / span
    return scaled if higher_is_better else 1 - scaled


display(HTML(
    """
    <style>
    .report-hero {
        padding: 22px 26px;
        border: 1px solid #d7dee8;
        border-radius: 8px;
        background: linear-gradient(135deg, #f8fbfc 0%, #ffffff 58%, #fff7ec 100%);
        margin-bottom: 18px;
    }
    .report-hero h1 {
        margin: 0 0 8px 0;
        font-size: 30px;
        letter-spacing: 0;
        color: #132530;
    }
    .report-hero p {
        margin: 4px 0;
        color: #344854;
        font-size: 14px;
    }
    .metric-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
        gap: 12px;
        margin: 12px 0 20px 0;
    }
    .metric-card {
        border: 1px solid #dce4ea;
        border-radius: 8px;
        padding: 14px 14px 12px 14px;
        background: #ffffff;
        border-top: 4px solid var(--accent);
    }
    .metric-card .label {
        color: #607080;
        font-size: 12px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0;
    }
    .metric-card .value {
        color: #12212b;
        font-size: 25px;
        line-height: 1.15;
        margin-top: 6px;
        font-weight: 800;
    }
    .metric-card .note {
        color: #647482;
        font-size: 12px;
        margin-top: 4px;
    }
    .insight-box {
        border-left: 5px solid #0E7C86;
        background: #f7fbfb;
        padding: 14px 18px;
        border-radius: 8px;
        margin: 12px 0 18px 0;
    }
    .warning-box {
        border-left: 5px solid #D95F59;
        background: #fff8f7;
        padding: 14px 18px;
        border-radius: 8px;
        margin: 12px 0 18px 0;
    }
    .small-muted {
        color: #607080;
        font-size: 12px;
    }
    </style>
    """
))


## 3. Load Dataset


In [3]:
DATA_PATH = Path("global_poverty_economic_inequality.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Cannot find source dataset: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

numeric_columns = df.select_dtypes(include="number").columns.tolist()
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.head()


,record_id,year,country,region,income_group,gdp_per_capita_usd,poverty_rate_pct,gini_coefficient,hdi_score,unemployment_rate_pct,inflation_rate_pct,literacy_rate_pct,life_expectancy_years,child_mortality_per_1000,electricity_access_pct,clean_water_access_pct,internet_penetration_pct,female_labor_participation_pct,social_protection_coverage_pct,income_share_top10_pct,income_share_bottom40_pct,urban_population_pct,remittances_pct_of_gdp,foreign_aid_million_usd,co2_per_capita_tonnes
0,POV0000001,2017,Bangladesh,South Asia,Lower-Middle Income,2985,23.19,39.22,0.69,5.46,12.39,79.80,66.70,45.00,58.20,86.10,60.70,13.30,40.00,46.50,10.50,58.40,5.80,124,1.78
1,POV0000002,2017,Cambodia,Southeast Asia,Lower-Middle Income,1651,36.72,34.04,0.69,7.01,3.38,55.20,68.30,55.90,73.70,85.10,39.90,44.80,43.10,47.90,17.70,34.40,12.80,28,1.49
2,POV0000003,2019,Tanzania,Sub-Saharan Africa,Low Income,1396,72.25,40.07,0.52,5.13,3.52,39.20,58.10,66.50,44.80,54.00,30.60,74.90,14.00,27.00,8.00,37.70,15.50,257,0.54
3,POV0000004,2022,Myanmar,Southeast Asia,Lower-Middle Income,1304,14.46,36.31,0.66,11.96,13.09,58.80,72.00,24.20,86.60,71.70,34.00,59.80,44.70,55.00,16.80,44.40,7.80,196,1.00
4,POV0000005,2017,Tanzania,Sub-Saharan Africa,Low Income,1290,41.03,44.10,0.41,6.98,12.33,35.30,54.20,111.80,37.10,55.20,10.50,54.30,24.70,45.20,15.60,25.50,24.60,946,0.49


## 4. Data Quality Snapshot

This step checks whether the dataset is ready for dashboarding and analytical modeling.


In [4]:
missing_cells = int(df.isna().sum().sum())
duplicate_record_ids = int(df["record_id"].duplicated().sum()) if "record_id" in df.columns else 0
year_min, year_max = int(df["year"].min()), int(df["year"].max())

quality_summary = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Countries",
            "Regions",
            "Income groups",
            "Year range",
            "Duplicate record_id",
            "Missing cells",
        ],
        "Value": [
            f"{len(df):,}",
            f"{df.shape[1]:,}",
            f"{df['country'].nunique():,}",
            f"{df['region'].nunique():,}",
            f"{df['income_group'].nunique():,}",
            f"{year_min} - {year_max}",
            f"{duplicate_record_ids:,}",
            f"{missing_cells:,}",
        ],
    }
)

display(
    quality_summary.style.hide(axis="index").set_table_styles(
        [
            {"selector": "th", "props": [("background-color", "#132530"), ("color", "white")]},
            {"selector": "td", "props": [("border-bottom", "1px solid #e6edf2")]},
        ]
    )
)

range_checks = pd.DataFrame(
    {
        "Column": [
            "poverty_rate_pct",
            "gini_coefficient",
            "hdi_score",
            "electricity_access_pct",
            "clean_water_access_pct",
            "internet_penetration_pct",
            "life_expectancy_years",
            "child_mortality_per_1000",
        ],
        "Min": [
            df["poverty_rate_pct"].min(),
            df["gini_coefficient"].min(),
            df["hdi_score"].min(),
            df["electricity_access_pct"].min(),
            df["clean_water_access_pct"].min(),
            df["internet_penetration_pct"].min(),
            df["life_expectancy_years"].min(),
            df["child_mortality_per_1000"].min(),
        ],
        "Median": [
            df["poverty_rate_pct"].median(),
            df["gini_coefficient"].median(),
            df["hdi_score"].median(),
            df["electricity_access_pct"].median(),
            df["clean_water_access_pct"].median(),
            df["internet_penetration_pct"].median(),
            df["life_expectancy_years"].median(),
            df["child_mortality_per_1000"].median(),
        ],
        "Max": [
            df["poverty_rate_pct"].max(),
            df["gini_coefficient"].max(),
            df["hdi_score"].max(),
            df["electricity_access_pct"].max(),
            df["clean_water_access_pct"].max(),
            df["internet_penetration_pct"].max(),
            df["life_expectancy_years"].max(),
            df["child_mortality_per_1000"].max(),
        ],
    }
)

display(range_checks.round(2).style.hide(axis="index"))


Metric,Value
Rows,"10,000"
Columns,25
Countries,40
Regions,10
Income groups,4
Year range,2015 - 2024
Duplicate record_id,0
Missing cells,0


Column,Min,Median,Max
poverty_rate_pct,0.500000,16.980000,74.870000
gini_coefficient,20.300000,42.080000,63.510000
hdi_score,0.350000,0.650000,0.990000
electricity_access_pct,20.000000,78.500000,100.000000
clean_water_access_pct,30.000000,78.900000,100.000000
internet_penetration_pct,5.000000,53.600000,99.000000
life_expectancy_years,52.000000,69.500000,86.300000
child_mortality_per_1000,1.500000,29.400000,119.700000


## 5. Feature Engineering

The following indicators make the dashboard more strategic:

- **Income concentration gap**: income share of top 10% minus bottom 40%.
- **Access foundation index**: combined electricity, clean water, internet, literacy, and social protection coverage.
- **Poverty pressure index**: combined pressure from poverty, unemployment, inflation, child mortality, weak access, low GDP, and low HDI.


In [5]:
df_model = df.copy()

df_model["income_concentration_gap"] = (
    df_model["income_share_top10_pct"] - df_model["income_share_bottom40_pct"]
)
df_model["access_foundation_index"] = df_model[
    [
        "electricity_access_pct",
        "clean_water_access_pct",
        "internet_penetration_pct",
        "literacy_rate_pct",
        "social_protection_coverage_pct",
    ]
].mean(axis=1)
df_model["human_capital_index"] = pd.concat(
    [
        minmax(df_model["hdi_score"]),
        minmax(df_model["literacy_rate_pct"]),
        minmax(df_model["life_expectancy_years"]),
        minmax(df_model["female_labor_participation_pct"]),
    ],
    axis=1,
).mean(axis=1) * 100
df_model["poverty_pressure_index"] = pd.concat(
    [
        minmax(df_model["poverty_rate_pct"]),
        minmax(df_model["unemployment_rate_pct"]),
        minmax(df_model["inflation_rate_pct"]),
        minmax(df_model["child_mortality_per_1000"]),
        minmax(np.log1p(df_model["gdp_per_capita_usd"]), higher_is_better=False),
        minmax(df_model["hdi_score"], higher_is_better=False),
        minmax(df_model["access_foundation_index"], higher_is_better=False),
    ],
    axis=1,
).mean(axis=1) * 100

df_model[
    [
        "country",
        "year",
        "region",
        "income_group",
        "poverty_rate_pct",
        "gini_coefficient",
        "income_concentration_gap",
        "access_foundation_index",
        "human_capital_index",
        "poverty_pressure_index",
    ]
].head()


,country,year,region,income_group,poverty_rate_pct,gini_coefficient,income_concentration_gap,access_foundation_index,human_capital_index,poverty_pressure_index
0,Bangladesh,2017,South Asia,Lower-Middle Income,23.19,39.22,36.00,64.96,43.79,41.52
1,Cambodia,2017,Southeast Asia,Lower-Middle Income,36.72,34.04,30.20,59.40,45.52,46.01
2,Tanzania,2019,Sub-Saharan Africa,Low Income,72.25,40.07,19.00,36.52,34.74,61.71
3,Myanmar,2022,Southeast Asia,Lower-Middle Income,14.46,36.31,38.20,59.16,53.30,45.71
4,Tanzania,2017,Sub-Saharan Africa,Low Income,41.03,44.10,29.60,32.56,19.73,68.66


## 6. Executive Dashboard


In [6]:
latest_year = int(df_model["year"].max())
latest = df_model[df_model["year"] == latest_year].copy()

country_latest = (
    latest.groupby(["country", "region", "income_group"], as_index=False)
    .agg(
        poverty_rate_pct=("poverty_rate_pct", "mean"),
        gini_coefficient=("gini_coefficient", "mean"),
        gdp_per_capita_usd=("gdp_per_capita_usd", "median"),
        hdi_score=("hdi_score", "mean"),
        unemployment_rate_pct=("unemployment_rate_pct", "mean"),
        inflation_rate_pct=("inflation_rate_pct", "mean"),
        literacy_rate_pct=("literacy_rate_pct", "mean"),
        life_expectancy_years=("life_expectancy_years", "mean"),
        child_mortality_per_1000=("child_mortality_per_1000", "mean"),
        electricity_access_pct=("electricity_access_pct", "mean"),
        clean_water_access_pct=("clean_water_access_pct", "mean"),
        internet_penetration_pct=("internet_penetration_pct", "mean"),
        female_labor_participation_pct=("female_labor_participation_pct", "mean"),
        social_protection_coverage_pct=("social_protection_coverage_pct", "mean"),
        income_concentration_gap=("income_concentration_gap", "mean"),
        access_foundation_index=("access_foundation_index", "mean"),
        human_capital_index=("human_capital_index", "mean"),
        poverty_pressure_index=("poverty_pressure_index", "mean"),
    )
)

region_snapshot = (
    df_model.groupby("region", as_index=False)
    .agg(
        records=("record_id", "count"),
        countries=("country", "nunique"),
        poverty_rate_pct=("poverty_rate_pct", "mean"),
        gini_coefficient=("gini_coefficient", "mean"),
        gdp_per_capita_usd=("gdp_per_capita_usd", "median"),
        hdi_score=("hdi_score", "mean"),
        child_mortality_per_1000=("child_mortality_per_1000", "mean"),
        electricity_access_pct=("electricity_access_pct", "mean"),
        clean_water_access_pct=("clean_water_access_pct", "mean"),
        internet_penetration_pct=("internet_penetration_pct", "mean"),
        social_protection_coverage_pct=("social_protection_coverage_pct", "mean"),
        access_foundation_index=("access_foundation_index", "mean"),
        poverty_pressure_index=("poverty_pressure_index", "mean"),
    )
    .sort_values("poverty_rate_pct", ascending=False)
)

highest_poverty_region = region_snapshot.iloc[0]
lowest_poverty_region = region_snapshot.iloc[-1]
highest_pressure_country = country_latest.sort_values("poverty_pressure_index", ascending=False).iloc[0]
lowest_pressure_country = country_latest.sort_values("poverty_pressure_index", ascending=True).iloc[0]

display(HTML(
    f"""
    <div class="report-hero">
        <h1>Global Poverty & Economic Inequality</h1>
        <p><strong>Created by Hieu Nguyen</strong></p>
        <p>Professional dashboard report covering {len(df_model):,} records, {df_model['country'].nunique():,} countries, {df_model['region'].nunique():,} regions, and years {year_min}-{year_max}.</p>
    </div>
    """
))

metrics = [
    ("Records", f"{len(df_model):,}", "Source observations", "#0E7C86"),
    ("Countries", f"{df_model['country'].nunique():,}", "Distinct economies", "#D95F59"),
    ("Latest year", str(latest_year), "Dashboard reference year", "#F2B84B"),
    ("Avg poverty", fmt_pct(df_model["poverty_rate_pct"].mean()), "Full dataset mean", "#5B5F97"),
    ("Avg Gini", fmt_num(df_model["gini_coefficient"].mean()), "Inequality intensity", "#2A9D8F"),
    ("Median GDP", fmt_money(df_model["gdp_per_capita_usd"].median()), "GDP per capita", "#8E6C88"),
    ("High-pressure country", highest_pressure_country["country"], f"{fmt_num(highest_pressure_country['poverty_pressure_index'])} index", "#607D8B"),
    ("Lowest-pressure country", lowest_pressure_country["country"], f"{fmt_num(lowest_pressure_country['poverty_pressure_index'])} index", "#E76F51"),
]

cards = ['<div class="metric-grid">']
for label, value, note, color in metrics:
    cards.append(
        f"""
        <div class="metric-card" style="--accent:{color}">
            <div class="label">{label}</div>
            <div class="value">{value}</div>
            <div class="note">{note}</div>
        </div>
        """
    )
cards.append("</div>")
display(HTML("".join(cards)))

display(HTML(
    f"""
    <div class="insight-box">
        <strong>Executive read:</strong> {highest_poverty_region['region']} has the highest average poverty rate at {fmt_pct(highest_poverty_region['poverty_rate_pct'])}, while {lowest_poverty_region['region']} has the lowest at {fmt_pct(lowest_poverty_region['poverty_rate_pct'])}. The dashboard should therefore be read as a portfolio of structural constraints, not a single poverty ranking.
    </div>
    """
))


## 7. Regional Snapshot


In [7]:
display(
    region_snapshot[
        [
            "region",
            "countries",
            "poverty_rate_pct",
            "gini_coefficient",
            "gdp_per_capita_usd",
            "hdi_score",
            "access_foundation_index",
            "social_protection_coverage_pct",
            "child_mortality_per_1000",
            "poverty_pressure_index",
        ]
    ]
    .round(2)
    .style.hide(axis="index")
    .background_gradient(subset=["poverty_rate_pct", "poverty_pressure_index", "child_mortality_per_1000"], cmap="Reds")
    .background_gradient(subset=["gdp_per_capita_usd", "hdi_score", "access_foundation_index", "social_protection_coverage_pct"], cmap="Greens")
    .format(
        {
            "poverty_rate_pct": "{:.1f}%",
            "gini_coefficient": "{:.1f}",
            "gdp_per_capita_usd": "${:,.0f}",
            "hdi_score": "{:.3f}",
            "access_foundation_index": "{:.1f}",
            "social_protection_coverage_pct": "{:.1f}%",
            "child_mortality_per_1000": "{:.1f}",
            "poverty_pressure_index": "{:.1f}",
        }
    )
)


region,countries,poverty_rate_pct,gini_coefficient,gdp_per_capita_usd,hdi_score,access_foundation_index,social_protection_coverage_pct,child_mortality_per_1000,poverty_pressure_index
Sub-Saharan Africa,12,37.8%,44.5,"$1,343",0.540,47.6,23.3%,59.8,55.8
Middle East & North Africa,3,32.2%,42.7,"$3,826",0.570,51.4,25.2%,50.6,51.4
South Asia,5,29.1%,42.2,"$2,420",0.580,53.4,26.4%,46.3,50.2
Southeast Asia,5,17.8%,42.9,"$3,987",0.650,64.6,36.8%,29.4,41.6
Latin America & Caribbean,5,13.7%,46.4,"$8,901",0.700,71.9,46.9%,23.5,36.1
Europe/Asia,1,6.5%,47.3,"$12,107",0.740,78.6,51.2%,12.1,30.4
East Asia,3,3.6%,39.6,"$17,536",0.830,86.3,66.1%,7.3,22.1
Europe,3,1.2%,32.4,"$52,410",0.930,94.0,80.1%,2.4,13.6
Oceania,1,1.1%,32.7,"$69,655",0.930,94.1,79.8%,2.2,12.9
North America,2,1.1%,32.5,"$79,480",0.930,94.1,79.9%,2.2,12.7


## 8. Macro Trends

This trend view separates poverty, inequality, income, and human development so the reader can detect whether economic improvement is broad-based.


In [8]:
year_trend = (
    df_model.groupby("year", as_index=False)
    .agg(
        poverty_rate_pct=("poverty_rate_pct", "mean"),
        gini_coefficient=("gini_coefficient", "mean"),
        gdp_per_capita_usd=("gdp_per_capita_usd", "median"),
        hdi_score=("hdi_score", "mean"),
        access_foundation_index=("access_foundation_index", "mean"),
        poverty_pressure_index=("poverty_pressure_index", "mean"),
    )
    .sort_values("year")
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Average Poverty Rate",
        "Average Gini Coefficient",
        "Median GDP per Capita",
        "Average HDI Score",
    ),
    vertical_spacing=0.14,
)
fig.add_trace(go.Scatter(x=year_trend["year"], y=year_trend["poverty_rate_pct"], mode="lines+markers", line=dict(color="#D95F59", width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=year_trend["year"], y=year_trend["gini_coefficient"], mode="lines+markers", line=dict(color="#5B5F97", width=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=year_trend["year"], y=year_trend["gdp_per_capita_usd"], mode="lines+markers", line=dict(color="#0E7C86", width=3)), row=2, col=1)
fig.add_trace(go.Scatter(x=year_trend["year"], y=year_trend["hdi_score"], mode="lines+markers", line=dict(color="#2A9D8F", width=3)), row=2, col=2)
fig.update_layout(height=650, showlegend=False, title_text="Macro Development Trends", title_x=0.02, margin=dict(l=35, r=20, t=80, b=35))
fig.update_yaxes(ticksuffix="%", row=1, col=1)
fig.update_yaxes(tickprefix="$", row=2, col=1)
fig.show()


## 9. Poverty, Growth, and Access Dashboard


In [9]:
fig = px.bar(
    region_snapshot.sort_values("poverty_rate_pct"),
    x="poverty_rate_pct",
    y="region",
    color="access_foundation_index",
    orientation="h",
    color_continuous_scale=["#2A9D8F", "#F2B84B", "#D95F59"],
    labels={
        "poverty_rate_pct": "Average poverty rate (%)",
        "region": "",
        "access_foundation_index": "Access foundation",
    },
    title="Regional Poverty Rate vs Access Foundation",
)
fig.update_layout(height=500, margin=dict(l=10, r=20, t=60, b=35), coloraxis_colorbar=dict(title="Access"))
fig.update_traces(hovertemplate="<b>%{y}</b><br>Poverty: %{x:.1f}%<br>Access index: %{marker.color:.1f}<extra></extra>")
fig.show()

fig = px.scatter(
    country_latest,
    x="gdp_per_capita_usd",
    y="poverty_rate_pct",
    color="region",
    size="child_mortality_per_1000",
    hover_name="country",
    hover_data={
        "income_group": True,
        "gini_coefficient": ":.1f",
        "hdi_score": ":.3f",
        "access_foundation_index": ":.1f",
        "gdp_per_capita_usd": ":,.0f",
        "poverty_rate_pct": ":.1f",
        "child_mortality_per_1000": ":.1f",
    },
    labels={
        "gdp_per_capita_usd": "GDP per capita, log scale",
        "poverty_rate_pct": "Poverty rate (%)",
        "child_mortality_per_1000": "Child mortality",
    },
    title=f"Latest-Year Country Positioning ({latest_year})",
)
fig.update_xaxes(type="log", tickprefix="$")
fig.update_layout(height=620, margin=dict(l=40, r=20, t=65, b=40), legend_title_text="Region")
fig.show()


## 10. Inequality and Distribution Signals


In [10]:
top_gap = country_latest.sort_values("income_concentration_gap", ascending=False).head(15)

fig = px.bar(
    top_gap.sort_values("income_concentration_gap"),
    x="income_concentration_gap",
    y="country",
    color="region",
    orientation="h",
    title=f"Countries with Highest Income Concentration Gap ({latest_year})",
    labels={"income_concentration_gap": "Top 10% share minus bottom 40% share", "country": ""},
)
fig.update_layout(height=560, margin=dict(l=15, r=20, t=65, b=40), legend_title_text="Region")
fig.show()

fig = px.scatter(
    country_latest,
    x="gini_coefficient",
    y="poverty_rate_pct",
    color="income_group",
    size="income_concentration_gap",
    hover_name="country",
    hover_data={"region": True, "access_foundation_index": ":.1f", "hdi_score": ":.3f"},
    title="Inequality, Poverty, and Income Group",
    labels={
        "gini_coefficient": "Gini coefficient",
        "poverty_rate_pct": "Poverty rate (%)",
        "income_concentration_gap": "Income concentration gap",
    },
)
fig.update_layout(height=590, margin=dict(l=40, r=20, t=65, b=40), legend_title_text="Income group")
fig.show()


## 11. Correlation Structure

Correlation helps identify where the strongest statistical signals sit. It does not prove causality, but it is useful for prioritizing questions.


In [11]:
corr_columns = [
    "gdp_per_capita_usd",
    "poverty_rate_pct",
    "gini_coefficient",
    "hdi_score",
    "unemployment_rate_pct",
    "inflation_rate_pct",
    "literacy_rate_pct",
    "life_expectancy_years",
    "child_mortality_per_1000",
    "electricity_access_pct",
    "clean_water_access_pct",
    "internet_penetration_pct",
    "female_labor_participation_pct",
    "social_protection_coverage_pct",
    "income_concentration_gap",
    "access_foundation_index",
    "poverty_pressure_index",
]
corr = df_model[corr_columns].corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Matrix: Poverty, Inequality, Access, and Human Development",
)
fig.update_layout(height=760, margin=dict(l=20, r=20, t=70, b=20))
fig.show()

poverty_corr = (
    corr["poverty_rate_pct"]
    .drop("poverty_rate_pct")
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .to_frame("Correlation with poverty")
)
display(poverty_corr.head(10).style.format("{:.3f}"))


,Correlation with poverty
poverty_pressure_index,0.889
access_foundation_index,-0.878
electricity_access_pct,-0.855
clean_water_access_pct,-0.848
child_mortality_per_1000,0.842
life_expectancy_years,-0.824
hdi_score,-0.812
literacy_rate_pct,-0.808
internet_penetration_pct,-0.802
social_protection_coverage_pct,-0.760


## 12. Country Rankings


In [12]:
rank_high_poverty = country_latest.sort_values("poverty_rate_pct", ascending=False).head(12)
rank_low_poverty = country_latest.sort_values("poverty_rate_pct", ascending=True).head(12)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Highest Poverty Rate", "Lowest Poverty Rate"), horizontal_spacing=0.18)
fig.add_trace(
    go.Bar(
        x=rank_high_poverty["poverty_rate_pct"],
        y=rank_high_poverty["country"],
        orientation="h",
        marker_color="#D95F59",
        hovertemplate="%{y}<br>Poverty: %{x:.1f}%<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=rank_low_poverty["poverty_rate_pct"],
        y=rank_low_poverty["country"],
        orientation="h",
        marker_color="#0E7C86",
        hovertemplate="%{y}<br>Poverty: %{x:.1f}%<extra></extra>",
    ),
    row=1,
    col=2,
)
fig.update_layout(height=620, showlegend=False, title_text=f"Country Poverty Ranking ({latest_year})", title_x=0.02)
fig.update_xaxes(ticksuffix="%")
fig.update_yaxes(autorange="reversed")
fig.show()


## 13. Strategic Segmentation

The clustering model groups countries by development pressure, not by geography alone. This is useful for policy prioritization because two countries in different regions can face a similar operating problem.


In [13]:
cluster_features = [
    "poverty_rate_pct",
    "gini_coefficient",
    "hdi_score",
    "unemployment_rate_pct",
    "inflation_rate_pct",
    "literacy_rate_pct",
    "life_expectancy_years",
    "child_mortality_per_1000",
    "electricity_access_pct",
    "clean_water_access_pct",
    "internet_penetration_pct",
    "social_protection_coverage_pct",
    "income_concentration_gap",
    "access_foundation_index",
    "poverty_pressure_index",
]
cluster_data = country_latest.dropna(subset=cluster_features).copy()
cluster_data["log_gdp_per_capita"] = np.log1p(cluster_data["gdp_per_capita_usd"])
model_features = cluster_features + ["log_gdp_per_capita"]

scaler = StandardScaler()
X = scaler.fit_transform(cluster_data[model_features])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=30)
cluster_data["cluster"] = kmeans.fit_predict(X)

cluster_profile = (
    cluster_data.groupby("cluster", as_index=False)
    .agg(
        countries=("country", "count"),
        poverty_rate_pct=("poverty_rate_pct", "mean"),
        gini_coefficient=("gini_coefficient", "mean"),
        gdp_per_capita_usd=("gdp_per_capita_usd", "median"),
        hdi_score=("hdi_score", "mean"),
        access_foundation_index=("access_foundation_index", "mean"),
        child_mortality_per_1000=("child_mortality_per_1000", "mean"),
        social_protection_coverage_pct=("social_protection_coverage_pct", "mean"),
        poverty_pressure_index=("poverty_pressure_index", "mean"),
    )
)

poverty_q = cluster_profile["poverty_rate_pct"].quantile([0.33, 0.66]).to_dict()
gdp_q = cluster_profile["gdp_per_capita_usd"].quantile([0.33, 0.66]).to_dict()
access_q = cluster_profile["access_foundation_index"].quantile([0.33, 0.66]).to_dict()
gini_q = cluster_profile["gini_coefficient"].quantile([0.33, 0.66]).to_dict()


def name_cluster(row: pd.Series) -> str:
    if row["poverty_rate_pct"] >= poverty_q[0.66] and row["gdp_per_capita_usd"] <= gdp_q[0.33]:
        return "High-poverty structural pressure"
    if row["poverty_rate_pct"] <= poverty_q[0.33] and row["gdp_per_capita_usd"] >= gdp_q[0.66]:
        return "Higher-income lower-poverty base"
    if row["gini_coefficient"] >= gini_q[0.66]:
        return "Inequality concentration risk"
    if row["access_foundation_index"] <= access_q[0.33]:
        return "Access foundation gap"
    return "Transition and stabilization group"


cluster_profile["cluster_label"] = cluster_profile.apply(name_cluster, axis=1)
label_map = cluster_profile.set_index("cluster")["cluster_label"].to_dict()
cluster_data["cluster_label"] = cluster_data["cluster"].map(label_map)

display(
    cluster_profile[
        [
            "cluster_label",
            "countries",
            "poverty_rate_pct",
            "gini_coefficient",
            "gdp_per_capita_usd",
            "hdi_score",
            "access_foundation_index",
            "child_mortality_per_1000",
            "social_protection_coverage_pct",
            "poverty_pressure_index",
        ]
    ]
    .sort_values("poverty_pressure_index", ascending=False)
    .round(2)
    .style.hide(axis="index")
    .format(
        {
            "poverty_rate_pct": "{:.1f}%",
            "gini_coefficient": "{:.1f}",
            "gdp_per_capita_usd": "${:,.0f}",
            "hdi_score": "{:.3f}",
            "access_foundation_index": "{:.1f}",
            "child_mortality_per_1000": "{:.1f}",
            "social_protection_coverage_pct": "{:.1f}%",
            "poverty_pressure_index": "{:.1f}",
        }
    )
)

fig = px.scatter(
    cluster_data,
    x="access_foundation_index",
    y="poverty_rate_pct",
    color="cluster_label",
    size="gdp_per_capita_usd",
    hover_name="country",
    hover_data={
        "region": True,
        "income_group": True,
        "gini_coefficient": ":.1f",
        "hdi_score": ":.3f",
        "poverty_pressure_index": ":.1f",
        "gdp_per_capita_usd": ":,.0f",
    },
    title=f"Development Archetypes by Access Foundation and Poverty ({latest_year})",
    labels={
        "access_foundation_index": "Access foundation index",
        "poverty_rate_pct": "Poverty rate (%)",
        "cluster_label": "Archetype",
    },
)
fig.update_layout(height=650, margin=dict(l=40, r=20, t=70, b=40), legend_title_text="Archetype")
fig.show()


  File "C:\Users\hieum\Desktop\Repo\Global_Poverty\Global_Poverty_and_Economic_Ineuqality\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


cluster_label,countries,poverty_rate_pct,gini_coefficient,gdp_per_capita_usd,hdi_score,access_foundation_index,child_mortality_per_1000,social_protection_coverage_pct,poverty_pressure_index
High-poverty structural pressure,11,50.3%,46.3,"$1,062",0.470,36.9,79.0,15.3%,63.3
Transition and stabilization group,13,19.8%,41.4,"$2,934",0.630,60.0,34.0,29.9%,42.7
Inequality concentration risk,8,4.6%,47.2,"$10,064",0.750,79.9,9.2,52.3%,27.9
Higher-income lower-poverty base,8,0.5%,32.7,"$60,452",0.940,94.8,1.5,80.5%,11.2


## 14. Driver Model for Analytical Prioritization

This ridge model estimates which standardized variables are most associated with poverty rates after considering the other features together. It is a prioritization model, not a causal claim.


In [14]:
driver_features = [
    "gini_coefficient",
    "income_concentration_gap",
    "unemployment_rate_pct",
    "inflation_rate_pct",
    "literacy_rate_pct",
    "life_expectancy_years",
    "child_mortality_per_1000",
    "electricity_access_pct",
    "clean_water_access_pct",
    "internet_penetration_pct",
    "female_labor_participation_pct",
    "social_protection_coverage_pct",
    "hdi_score",
    "access_foundation_index",
]
driver_data = df_model.dropna(subset=driver_features + ["poverty_rate_pct", "gdp_per_capita_usd"]).copy()
driver_data["log_gdp_per_capita"] = np.log1p(driver_data["gdp_per_capita_usd"])
driver_features = ["log_gdp_per_capita"] + driver_features

X = driver_data[driver_features]
y = driver_data["poverty_rate_pct"]
X_scaled = StandardScaler().fit_transform(X)
ridge = Ridge(alpha=2.0)
ridge.fit(X_scaled, y)

coefficients = (
    pd.DataFrame({"Feature": driver_features, "Standardized coefficient": ridge.coef_})
    .assign(abs_coef=lambda data: data["Standardized coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)
display(coefficients.head(12).style.hide(axis="index").format({"Standardized coefficient": "{:+.3f}"}))

fig = px.bar(
    coefficients.head(12).sort_values("Standardized coefficient"),
    x="Standardized coefficient",
    y="Feature",
    orientation="h",
    color="Standardized coefficient",
    color_continuous_scale=["#0E7C86", "#f7f7f7", "#D95F59"],
    title="Top Poverty-Associated Drivers in Standardized Ridge Model",
)
fig.update_layout(height=560, margin=dict(l=20, r=20, t=65, b=35), coloraxis_showscale=False)
fig.show()


Feature,Standardized coefficient
electricity_access_pct,-4.627
child_mortality_per_1000,+4.146
clean_water_access_pct,-3.909
life_expectancy_years,-2.193
access_foundation_index,-2.190
literacy_rate_pct,-1.957
hdi_score,-1.866
log_gdp_per_capita,+1.287
internet_penetration_pct,-1.283
social_protection_coverage_pct,+1.216


## 15. Deep Insights and Strategic Recommendations


In [15]:
poverty_corr_sorted = corr["poverty_rate_pct"].drop("poverty_rate_pct").sort_values()
strongest_negative = poverty_corr_sorted.head(4)
strongest_positive = poverty_corr_sorted.tail(4).sort_values(ascending=False)

trend_first = year_trend.iloc[0]
trend_last = year_trend.iloc[-1]
poverty_change = trend_last["poverty_rate_pct"] - trend_first["poverty_rate_pct"]
gini_change = trend_last["gini_coefficient"] - trend_first["gini_coefficient"]
access_change = trend_last["access_foundation_index"] - trend_first["access_foundation_index"]

pressure_regions = region_snapshot.sort_values("poverty_pressure_index", ascending=False).head(3)
best_access_regions = region_snapshot.sort_values("access_foundation_index", ascending=False).head(3)

driver_top = coefficients.iloc[0]
driver_second = coefficients.iloc[1]

def corr_sentence(series: pd.Series) -> str:
    return ", ".join([f"{name} ({value:+.2f})" for name, value in series.items()])


insight_md = f"""
### Analyst view

1. **Poverty is a systems outcome, not only an income outcome.** The strongest negative poverty correlations are {corr_sentence(strongest_negative)}. This suggests that development strategy should combine income growth with health, education, and basic access rather than treating GDP as the only dashboard target.

2. **The most visible pressure signals move through deprivation and fragility.** The strongest positive poverty correlations are {corr_sentence(strongest_positive)}. When child mortality, inequality concentration, unemployment, or weak access rise together, poverty risk compounds.

3. **Trend direction matters.** From {int(trend_first['year'])} to {int(trend_last['year'])}, average poverty changed by {poverty_change:+.1f} percentage points, average Gini changed by {gini_change:+.1f}, and the access foundation index changed by {access_change:+.1f}. A good poverty strategy should therefore monitor whether access gains are large enough to offset macro and inequality pressures.

4. **Regional prioritization should use pressure plus feasibility.** The highest poverty-pressure regions are {', '.join(pressure_regions['region'].tolist())}. The strongest access foundations are {', '.join(best_access_regions['region'].tolist())}. Regions with high pressure and weak access need foundational investment; regions with decent access but persistent inequality need distribution-sensitive growth.

5. **The driver model flags where to investigate first.** The largest standardized association is `{driver_top['Feature']}` ({driver_top['Standardized coefficient']:+.3f}), followed by `{driver_second['Feature']}` ({driver_second['Standardized coefficient']:+.3f}). These should be read as priority diagnostics for deeper country-level analysis.

### Strategic recommendations

- **Build an access floor:** prioritize electricity, clean water, internet, literacy, and social protection together. The dashboard consistently shows that poverty is lower where the access foundation is stronger.
- **Pair growth with distribution:** GDP improvement without better bottom-40 income share can leave poverty reduction fragile. Track the income concentration gap next to Gini.
- **Treat child mortality as an early warning indicator:** it is both a welfare outcome and a signal that household vulnerability is deep.
- **Segment interventions by archetype:** high-pressure countries need basic service and safety-net expansion; inequality-risk countries need inclusive labor markets and distribution policy; higher-income lower-poverty countries should focus on resilience and targeted pockets of deprivation.
- **Use the dashboard as a monitoring system:** refresh the dataset annually, watch poverty pressure index movement, and inspect countries that improve GDP but not access or inequality.
"""

display(Markdown(insight_md))
display(HTML('<p class="small-muted">Created by Hieu Nguyen</p>'))



### Analyst view

1. **Poverty is a systems outcome, not only an income outcome.** The strongest negative poverty correlations are access_foundation_index (-0.88), electricity_access_pct (-0.86), clean_water_access_pct (-0.85), life_expectancy_years (-0.82). This suggests that development strategy should combine income growth with health, education, and basic access rather than treating GDP as the only dashboard target.

2. **The most visible pressure signals move through deprivation and fragility.** The strongest positive poverty correlations are poverty_pressure_index (+0.89), child_mortality_per_1000 (+0.84), gini_coefficient (+0.33), income_concentration_gap (+0.00). When child mortality, inequality concentration, unemployment, or weak access rise together, poverty risk compounds.

3. **Trend direction matters.** From 2015 to 2024, average poverty changed by -4.5 percentage points, average Gini changed by +0.3, and the access foundation index changed by +2.6. A good poverty strategy should therefore monitor whether access gains are large enough to offset macro and inequality pressures.

4. **Regional prioritization should use pressure plus feasibility.** The highest poverty-pressure regions are Sub-Saharan Africa, Middle East & North Africa, South Asia. The strongest access foundations are Oceania, North America, Europe. Regions with high pressure and weak access need foundational investment; regions with decent access but persistent inequality need distribution-sensitive growth.

5. **The driver model flags where to investigate first.** The largest standardized association is `electricity_access_pct` (-4.627), followed by `child_mortality_per_1000` (+4.146). These should be read as priority diagnostics for deeper country-level analysis.

### Strategic recommendations

- **Build an access floor:** prioritize electricity, clean water, internet, literacy, and social protection together. The dashboard consistently shows that poverty is lower where the access foundation is stronger.
- **Pair growth with distribution:** GDP improvement without better bottom-40 income share can leave poverty reduction fragile. Track the income concentration gap next to Gini.
- **Treat child mortality as an early warning indicator:** it is both a welfare outcome and a signal that household vulnerability is deep.
- **Segment interventions by archetype:** high-pressure countries need basic service and safety-net expansion; inequality-risk countries need inclusive labor markets and distribution policy; higher-income lower-poverty countries should focus on resilience and targeted pockets of deprivation.
- **Use the dashboard as a monitoring system:** refresh the dataset annually, watch poverty pressure index movement, and inspect countries that improve GDP but not access or inequality.


## 16. Reproducibility Checklist

- Source file: `global_poverty_economic_inequality.csv`
- Environment folder: `venv`
- Dependency file: `requirements.txt`
- Notebook kernel: `Python (Global Poverty venv)`
- Report author line: `Created by Hieu Nguyen`
